# LeetCode 1670 — Design Front Middle Back Queue bằng Python

Link đề: https://leetcode.com/problems/design-front-middle-back-queue/ · Độ khó: **Medium** · Chủ đề: Array, Linked List, Design, Queue, Data Stream

Skeleton LeetCode:

```python
class FrontMiddleBackQueue:

    def __init__(self):
        ...

    def pushFront(self, val: int) -> None:
        ...

    def pushMiddle(self, val: int) -> None:
        ...

    def pushBack(self, val: int) -> None:
        ...

    def popFront(self) -> int:
        ...

    def popMiddle(self) -> int:
        ...

    def popBack(self) -> int:
        ...
```

Đây là bài **thiết kế lớp**: LeetCode gọi một chuỗi lệnh và so sánh kết quả từng lệnh. Notebook dựng sẵn **bộ chạy chuỗi lệnh** giống LeetCode và một cài đặt **brute force bằng list** để đối chiếu.

## 1. Mục tiêu học

1. Đọc hiểu một bài **design**: input là hai mảng (tên lệnh, tham số), output là mảng kết quả có `null` cho các lệnh `void`.
2. Xác định chính xác **chỉ số "giữa"** khi chèn (`n // 2`) và khi lấy ra (`(n − 1) // 2`) với quy ước "chọn vị trí giữa **phía trước**".
3. Cài đặt baseline bằng `list` (`O(n)` cho thao tác giữa), rồi tối ưu bằng **hai `deque` cân bằng** (`O(1)` mỗi thao tác) với một **bất biến kích thước** rõ ràng.
4. Cài đặt nâng cao bằng **danh sách liên kết đôi + con trỏ giữa** và tự chứng minh quy tắc dịch con trỏ.
5. Kiểm thử bằng chuỗi lệnh ngẫu nhiên có seed cố định, đối chiếu từng kết quả với brute force.

## 2. Đề bài & ví dụ

Thiết kế hàng đợi hỗ trợ thêm/lấy ở **đầu**, **giữa** và **cuối**. Cài đặt lớp `FrontMiddleBackQueue`:
- `FrontMiddleBackQueue()` khởi tạo hàng đợi rỗng.
- `pushFront(val)` thêm `val` vào **đầu**.
- `pushMiddle(val)` thêm `val` vào **giữa**.
- `pushBack(val)` thêm `val` vào **cuối**.
- `popFront()` xóa và trả phần tử **đầu**; nếu rỗng trả `-1`.
- `popMiddle()` xóa và trả phần tử **giữa**; nếu rỗng trả `-1`.
- `popBack()` xóa và trả phần tử **cuối**; nếu rỗng trả `-1`.

**Lưu ý:** khi có **hai** vị trí giữa, thao tác thực hiện ở vị trí giữa **phía trước**. Ví dụ:
- Thêm `6` vào giữa `[1,2,3,4,5]` → `[1,2,6,3,4,5]`.
- Lấy giữa từ `[1,2,3,4,5,6]` → trả `3`, còn `[1,2,4,5,6]`.

**Ví dụ 1:**

```
Input:  ["FrontMiddleBackQueue","pushFront","pushBack","pushMiddle","pushMiddle","popFront","popMiddle","popMiddle","popBack","popFront"]
        [[],[1],[2],[3],[4],[],[],[],[],[]]
Output: [null,null,null,null,null,1,3,4,2,-1]
```

Diễn giải: `[1]` → `[1,2]` → `[1,3,2]` → `[1,4,3,2]` → popFront trả 1 → `[4,3,2]` → popMiddle trả 3 → `[4,2]` → popMiddle trả 4 → `[2]` → popBack trả 2 → `[]` → popFront trả −1.

**Ràng buộc:**
- `1 ≤ val ≤ 10⁹`
- Tối đa `1000` lời gọi tới các hàm `push*`/`pop*`.

In [1]:
import random
import time
from collections import deque
from typing import List

def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài design: lệnh đầu là constructor, trả mảng kết quả."""
    out = []
    obj = None
    for op, arg in zip(ops, args):
        if op == cls.__name__ or op == "FrontMiddleBackQueue":
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

EX_OPS = ["FrontMiddleBackQueue", "pushFront", "pushBack", "pushMiddle", "pushMiddle",
          "popFront", "popMiddle", "popMiddle", "popBack", "popFront"]
EX_ARGS = [[], [1], [2], [3], [4], [], [], [], [], []]
EX_OUT = [None, None, None, None, None, 1, 3, 4, 2, -1]
print("số lệnh:", len(EX_OPS), "| kỳ vọng:", EX_OUT)

số lệnh: 10 | kỳ vọng: [None, None, None, None, None, 1, 3, 4, 2, -1]


## 3. Kiến thức nền

### 3.1 `list.insert` / `list.pop(i)` — tiện nhưng `O(n)`

`list` của Python là mảng động: chèn/xóa ở vị trí `i` phải dịch mọi phần tử phía sau ⇒ `O(n − i)`. Chỉ `append`/`pop()` ở cuối là `O(1)`.

In [2]:
a = [1, 2, 3, 4, 5]
a.insert(len(a) // 2, 6)       # chèn vào giữa (phía trước)
print("chèn 6 vào giữa:", a)
x = a.pop((len(a) - 1) // 2)  # lấy phần tử giữa (phía trước)
print("lấy giữa:", x, "| còn lại:", a)
a.insert(0, 0); print("chèn đầu:", a, "| pop(0):", a.pop(0), a)

chèn 6 vào giữa: [1, 2, 6, 3, 4, 5]
lấy giữa: 6 | còn lại: [1, 2, 3, 4, 5]
chèn đầu: [1, 2, 3, 4, 5] | pop(0): 0 [1, 2, 3, 4, 5]


### 3.2 `collections.deque` — thêm/bớt hai đầu `O(1)`

`deque` hỗ trợ `append`, `appendleft`, `pop`, `popleft` đều `O(1)`. Truy cập/chèn ở giữa vẫn chậm, nên ta sẽ dùng **hai** deque ghép lại, "giữa" chính là **chỗ nối** của chúng.

In [3]:
d = deque([2, 3])
d.appendleft(1); d.append(4)
print(d, "| popleft:", d.popleft(), "| pop:", d.pop(), "| còn:", d)

deque([2, 3]) | popleft: 1 | pop: 4 | còn: deque([2, 3])


### 3.3 Công thức chỉ số giữa

Với `n` phần tử (chỉ số `0..n−1`):
- **Lấy giữa** (phía trước): chỉ số `(n − 1) // 2`. `n = 6` → `2`; `n = 5` → `2`; `n = 1` → `0`.
- **Chèn giữa** (phía trước): chèn vào chỉ số `n // 2` (sau khi chèn, phần tử mới đứng ở đó). `n = 5` → `2`; `n = 4` → `2`; `n = 0` → `0`.

Kiểm tra lại: chèn vào `n` phần tử tạo mảng `n+1` phần tử, phần tử mới phải là "giữa phía trước" của mảng mới: `((n+1) − 1) // 2 = n // 2` ✓.

In [4]:
for n in range(0, 8):
    arr = list(range(n))
    pop_idx = (n - 1) // 2 if n else None
    ins = arr[:]; ins.insert(n // 2, "X")
    print(f"n={n}: lấy giữa ở chỉ số {pop_idx}, chèn giữa -> {ins} (X ở chỉ số {ins.index('X')} = ({n}+1-1)//2)")

n=0: lấy giữa ở chỉ số None, chèn giữa -> ['X'] (X ở chỉ số 0 = (0+1-1)//2)
n=1: lấy giữa ở chỉ số 0, chèn giữa -> ['X', 0] (X ở chỉ số 0 = (1+1-1)//2)
n=2: lấy giữa ở chỉ số 0, chèn giữa -> [0, 'X', 1] (X ở chỉ số 1 = (2+1-1)//2)
n=3: lấy giữa ở chỉ số 1, chèn giữa -> [0, 'X', 1, 2] (X ở chỉ số 1 = (3+1-1)//2)
n=4: lấy giữa ở chỉ số 1, chèn giữa -> [0, 1, 'X', 2, 3] (X ở chỉ số 2 = (4+1-1)//2)
n=5: lấy giữa ở chỉ số 2, chèn giữa -> [0, 1, 'X', 2, 3, 4] (X ở chỉ số 2 = (5+1-1)//2)
n=6: lấy giữa ở chỉ số 2, chèn giữa -> [0, 1, 2, 'X', 3, 4, 5] (X ở chỉ số 3 = (6+1-1)//2)
n=7: lấy giữa ở chỉ số 3, chèn giữa -> [0, 1, 2, 'X', 3, 4, 5, 6] (X ở chỉ số 3 = (7+1-1)//2)


### 3.4 Bài design trên LeetCode chạy thế nào?

LeetCode đưa hai mảng: `ops[i]` là tên phương thức, `args[i]` là danh sách tham số. Lệnh đầu tiên là **constructor**. Mỗi phương thức trả `None` (→ `null`) hoặc một giá trị. Hàm `run_ops` ở trên mô phỏng đúng điều đó bằng `getattr(obj, op)(*arg)`.

In [5]:
class Toy:
    def __init__(self): self.s = 0
    def add(self, v): self.s += v
    def get(self): return self.s

print(run_ops(Toy, ["Toy", "add", "add", "get"], [[], [3], [4], []]))

[None, None, None, 7]


## 4. Ý tưởng nền tảng / trực giác — tính bằng tay

Chạy ví dụ 1, ghi lại hàng đợi và **vị trí giữa** sau mỗi lệnh:

| lệnh | trước | chỉ số thao tác | sau | trả về |
|---|---|---|---|---|
| pushFront(1) | `[]` | 0 | `[1]` | null |
| pushBack(2) | `[1]` | cuối | `[1,2]` | null |
| pushMiddle(3) | `[1,2]` | `2 // 2 = 1` | `[1,3,2]` | null |
| pushMiddle(4) | `[1,3,2]` | `3 // 2 = 1` | `[1,4,3,2]` | null |
| popFront() | `[1,4,3,2]` | 0 | `[4,3,2]` | 1 |
| popMiddle() | `[4,3,2]` | `(3−1)//2 = 1` | `[4,2]` | 3 |
| popMiddle() | `[4,2]` | `(2−1)//2 = 0` | `[2]` | 4 |
| popBack() | `[2]` | cuối | `[]` | 2 |
| popFront() | `[]` | — | `[]` | −1 |

**Trực giác cho cách tối ưu:** cắt hàng đợi làm hai nửa `L | R`. Nếu giữ `L` luôn chứa "nửa trước" (kể cả phần tử giữa) thì:
- phần tử giữa phía trước luôn là **phần tử cuối của `L`**,
- chèn giữa ≈ thêm vào **cuối `L`** (hoặc đầu `R`),
- mọi thao tác chỉ chạm **đầu/cuối** của `L`, `R` ⇒ dùng `deque` là `O(1)`.

## 5. Cách 1 — Brute Force: một `list` Python

**Ý tưởng:** dùng đúng công thức chỉ số ở mục 3.3 với `list.insert` / `list.pop(i)`.

Đây cũng là **cài đặt chuẩn** (oracle) để kiểm thử các cách nhanh hơn.

In [6]:
class QueueList:
    def __init__(self):
        self.a = []
    def pushFront(self, val: int) -> None:
        self.a.insert(0, val)
    def pushMiddle(self, val: int) -> None:
        self.a.insert(len(self.a) // 2, val)
    def pushBack(self, val: int) -> None:
        self.a.append(val)
    def popFront(self) -> int:
        return self.a.pop(0) if self.a else -1
    def popMiddle(self) -> int:
        return self.a.pop((len(self.a) - 1) // 2) if self.a else -1
    def popBack(self) -> int:
        return self.a.pop() if self.a else -1

got = run_ops(QueueList, EX_OPS, EX_ARGS)
print(got)
assert got == EX_OUT

[None, None, None, None, None, 1, 3, 4, 2, -1]


**Độ phức tạp Cách 1:** `pushFront`, `pushMiddle`, `popFront`, `popMiddle` là `O(n)` (dịch phần tử); `pushBack`, `popBack` là `O(1)`. Space `O(n)`.

Với tối đa 1000 lời gọi thì `O(n)` mỗi lệnh vẫn qua (≈ `10⁶` thao tác), nhưng nếu số lệnh lên `10⁵`–`10⁶` thì sẽ chậm rõ rệt:

**Nút thắt:** chèn/xóa ở **giữa** mảng. Đo:

In [7]:
def bench(cls, n_ops, seed=0):
    rnd = random.Random(seed)
    q = cls()
    t = time.perf_counter()
    for _ in range(n_ops):
        if rnd.random() < 0.7:
            q.pushMiddle(rnd.randint(1, 10**9))
        else:
            q.popMiddle()
    return time.perf_counter() - t

for n_ops in (10**4, 4 * 10**4, 10**5):
    print(f"QueueList, {n_ops:6} lệnh giữa: {bench(QueueList, n_ops) * 1000:7.1f} ms")

QueueList,  10000 lệnh giữa:     3.7 ms
QueueList,  40000 lệnh giữa:    29.7 ms


QueueList, 100000 lệnh giữa:   157.7 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** chia hàng đợi thành hai deque `L` (nửa trước) và `R` (nửa sau), hàng đợi = `L + R`, và giữ **bất biến kích thước**:

> `len(L) == len(R)` **hoặc** `len(L) == len(R) + 1`.

Khi đó với `n = len(L) + len(R)`:
- Nếu `n` chẵn: `len(L) = n/2`, phần tử giữa phía trước có chỉ số `n/2 − 1` = **cuối `L`**.
- Nếu `n` lẻ: `len(L) = (n+1)/2`, phần tử giữa có chỉ số `(n−1)/2` = **cuối `L`**.

⇒ **`popMiddle` luôn là `L.pop()`** — không cần xét chẵn lẻ!

**Quy tắc "nếu… thì…":**
- `pushFront(v)`: `L.appendleft(v)` rồi cân bằng.
- `pushBack(v)`: `R.append(v)` rồi cân bằng.
- `pushMiddle(v)`: chèn vào chỉ số `n // 2`. Nếu `len(L) > len(R)` (n lẻ) **thì** chỉ số đó là ngay **trước** phần tử cuối của `L` ⇒ chuyển cuối `L` sang đầu `R` trước, rồi `L.append(v)`. Ngược lại (n chẵn) **thì** chỉ số đó là ngay sau `L` ⇒ `L.append(v)`.
- `popFront`: lấy `L.popleft()` (nếu `L` rỗng thì cả hàng rỗng, vì `len(L) ≥ len(R)`), rồi cân bằng.
- `popBack`: nếu `R` rỗng **thì** phần tử cuối nằm ở `L` (chỉ khi `n = 1`), lấy `L.pop()`; ngược lại `R.pop()`; rồi cân bằng.
- **Cân bằng:** nếu `len(L) > len(R) + 1` thì chuyển cuối `L` sang đầu `R`; nếu `len(R) > len(L)` thì chuyển đầu `R` sang cuối `L`. Mỗi thao tác chỉ làm lệch tối đa 1 ⇒ cân bằng tối đa 1 lần di chuyển.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai `deque` cân bằng (middle luôn ở cuối `L`)

**Từng đoạn:**
1. `__init__`: `self.L = deque()`, `self.R = deque()`.
2. `_balance()`: sửa lệch theo bất biến.
3. Mỗi thao tác theo bảng quy tắc ở mục 6.

Code demo — in `L | R` sau mỗi lệnh của ví dụ 1:

In [8]:
class QueueTwoDequeDemo:
    def __init__(self):
        self.L, self.R = deque(), deque()
    def _balance(self):
        if len(self.L) > len(self.R) + 1:
            self.R.appendleft(self.L.pop())
        elif len(self.R) > len(self.L):
            self.L.append(self.R.popleft())
    def show(self, tag, ret=None):
        print(f"{tag:16} L={list(self.L)!s:14} R={list(self.R)!s:10} trả={ret}")
    def pushFront(self, v): self.L.appendleft(v); self._balance(); self.show(f"pushFront({v})")
    def pushBack(self, v): self.R.append(v); self._balance(); self.show(f"pushBack({v})")
    def pushMiddle(self, v):
        if len(self.L) > len(self.R):
            self.R.appendleft(self.L.pop())
        self.L.append(v); self.show(f"pushMiddle({v})")
    def popFront(self):
        r = -1
        if self.L:
            r = self.L.popleft(); self._balance()
        self.show("popFront()", r); return r
    def popMiddle(self):
        r = -1
        if self.L:
            r = self.L.pop(); self._balance()
        self.show("popMiddle()", r); return r
    def popBack(self):
        r = -1
        if self.R: r = self.R.pop(); self._balance()
        elif self.L: r = self.L.pop()
        self.show("popBack()", r); return r

assert run_ops(QueueTwoDequeDemo, EX_OPS, EX_ARGS) == EX_OUT

pushFront(1)     L=[1]            R=[]         trả=None
pushBack(2)      L=[1]            R=[2]        trả=None
pushMiddle(3)    L=[1, 3]         R=[2]        trả=None
pushMiddle(4)    L=[1, 4]         R=[3, 2]     trả=None
popFront()       L=[4, 3]         R=[2]        trả=1
popMiddle()      L=[4]            R=[2]        trả=3
popMiddle()      L=[2]            R=[]         trả=4
popBack()        L=[]             R=[]         trả=2
popFront()       L=[]             R=[]         trả=-1


Code nộp LeetCode của Cách 2:

In [9]:
class QueueTwoDeque:
    def __init__(self):
        self.L = deque()    # nửa trước, len(L) == len(R) hoặc len(R) + 1
        self.R = deque()    # nửa sau

    def _balance(self) -> None:
        if len(self.L) > len(self.R) + 1:
            self.R.appendleft(self.L.pop())
        elif len(self.R) > len(self.L):
            self.L.append(self.R.popleft())

    def pushFront(self, val: int) -> None:
        self.L.appendleft(val)
        self._balance()

    def pushMiddle(self, val: int) -> None:
        if len(self.L) > len(self.R):        # n lẻ: chèn trước phần tử cuối của L
            self.R.appendleft(self.L.pop())
        self.L.append(val)

    def pushBack(self, val: int) -> None:
        self.R.append(val)
        self._balance()

    def popFront(self) -> int:
        if not self.L:
            return -1
        val = self.L.popleft()
        self._balance()
        return val

    def popMiddle(self) -> int:
        if not self.L:
            return -1
        val = self.L.pop()                   # giữa phía trước luôn là cuối L
        self._balance()
        return val

    def popBack(self) -> int:
        if self.R:
            val = self.R.pop()
            self._balance()
            return val
        if self.L:                           # chỉ còn 1 phần tử, nằm ở L
            return self.L.pop()
        return -1

assert run_ops(QueueTwoDeque, EX_OPS, EX_ARGS) == EX_OUT
print("Cách 2 qua ví dụ đề")

Cách 2 qua ví dụ đề


**Độ phức tạp Cách 2:** mọi thao tác `O(1)` (mỗi lệnh tối đa một lần di chuyển phần tử giữa hai deque). Space `O(n)`.

### 7.2 Cách 3 — Danh sách liên kết đôi + con trỏ `mid`

Một cách thuần "con trỏ": danh sách liên kết đôi có hai nút canh `head`, `tail`, cộng **con trỏ `mid`** luôn trỏ vào phần tử chỉ số `(n − 1) // 2`. Mỗi thao tác chèn/xóa `O(1)` rồi **dịch `mid` tối đa một bước**.

Quy tắc dịch `mid` (`n` là kích thước **trước** thao tác; tự kiểm bằng cách so chỉ số cũ/mới):

| thao tác | `n` chẵn | `n` lẻ |
|---|---|---|
| pushFront | giữ nguyên | `mid = mid.prev` |
| pushBack | `mid = mid.next` | giữ nguyên |
| pushMiddle | chèn **sau** `mid`, `mid = mới` | chèn **trước** `mid`, `mid = mới` |
| popFront | `mid = mid.next` | giữ nguyên |
| popBack | giữ nguyên | `mid = mid.prev` |
| popMiddle | `mid = mid.next` | `mid = mid.prev` |

(Trường hợp `n = 0` khi push: `mid` = nút mới; khi pop còn 0 phần tử: `mid = None`.)

Ví dụ kiểm tra dòng pushFront, `n` lẻ `= 2k+1`: `mid` cũ ở chỉ số `k`; sau khi thêm đầu, nút đó lùi tới chỉ số `k+1`, còn chỉ số giữa mới là `(2k+2−1)//2 = k` ⇒ phải lùi `mid` về `mid.prev`. ✓

In [10]:
class DNode:
    __slots__ = ("val", "prev", "next")
    def __init__(self, val=0):
        self.val = val; self.prev = None; self.next = None

class QueueLinked:
    def __init__(self):
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head
        self.n = 0
        self.mid = None

    def _insert_after(self, node, val):
        x = DNode(val)
        x.prev, x.next = node, node.next
        node.next.prev = x
        node.next = x
        self.n += 1
        return x

    def _remove(self, x):
        x.prev.next, x.next.prev = x.next, x.prev
        self.n -= 1
        return x.val

    def pushFront(self, val: int) -> None:
        n = self.n
        x = self._insert_after(self.head, val)
        if n == 0: self.mid = x
        elif n % 2 == 1: self.mid = self.mid.prev

    def pushBack(self, val: int) -> None:
        n = self.n
        x = self._insert_after(self.tail.prev, val)
        if n == 0: self.mid = x
        elif n % 2 == 0: self.mid = self.mid.next

    def pushMiddle(self, val: int) -> None:
        n = self.n
        if n == 0:
            self.mid = self._insert_after(self.head, val)
        elif n % 2 == 0:
            self.mid = self._insert_after(self.mid, val)
        else:
            self.mid = self._insert_after(self.mid.prev, val)

    def popFront(self) -> int:
        if self.n == 0: return -1
        n = self.n
        x = self.head.next
        if n == 1: self.mid = None
        elif n % 2 == 0: self.mid = self.mid.next
        return self._remove(x)

    def popBack(self) -> int:
        if self.n == 0: return -1
        n = self.n
        x = self.tail.prev
        if n == 1: self.mid = None
        elif n % 2 == 1: self.mid = self.mid.prev
        return self._remove(x)

    def popMiddle(self) -> int:
        if self.n == 0: return -1
        n, x = self.n, self.mid
        if n == 1: self.mid = None
        elif n % 2 == 0: self.mid = x.next
        else: self.mid = x.prev
        return self._remove(x)

    def to_list(self):
        out, c = [], self.head.next
        while c is not self.tail:
            out.append(c.val); c = c.next
        return out

assert run_ops(QueueLinked, EX_OPS, EX_ARGS) == EX_OUT
print("Cách 3 qua ví dụ đề")

Cách 3 qua ví dụ đề


Demo: kiểm tra `mid` luôn trỏ đúng phần tử chỉ số `(n−1)//2` sau mỗi thao tác của một chuỗi ngẫu nhiên ngắn:

In [11]:
rnd = random.Random(5)
q, ref = QueueLinked(), QueueList()
names = ["pushFront", "pushMiddle", "pushBack", "popFront", "popMiddle", "popBack"]
for step in range(12):
    op = rnd.choice(names)
    arg = [rnd.randint(1, 99)] if op.startswith("push") else []
    r1, r2 = getattr(q, op)(*arg), getattr(ref, op)(*arg)
    cur = q.to_list()
    mid_val = q.mid.val if q.mid else None
    exp_mid = cur[(len(cur) - 1) // 2] if cur else None
    print(f"{op + str(arg):18} -> {cur!s:28} mid={mid_val!s:5} (đúng: {exp_mid})")
    assert r1 == r2 and cur == ref.a and mid_val == exp_mid

popMiddle[]        -> []                           mid=None  (đúng: None)
pushBack[95]       -> [95]                         mid=95    (đúng: 95)
pushBack[89]       -> [95, 89]                     mid=95    (đúng: 95)
popBack[]          -> [95]                         mid=95    (đúng: 95)
popBack[]          -> []                           mid=None  (đúng: None)
popMiddle[]        -> []                           mid=None  (đúng: None)
pushFront[60]      -> [60]                         mid=60    (đúng: 60)
pushMiddle[84]     -> [84, 60]                     mid=84    (đúng: 84)
pushFront[21]      -> [21, 84, 60]                 mid=84    (đúng: 84)
pushFront[48]      -> [48, 21, 84, 60]             mid=21    (đúng: 21)
popFront[]         -> [21, 84, 60]                 mid=84    (đúng: 84)
pushMiddle[49]     -> [21, 49, 84, 60]             mid=49    (đúng: 49)


**Độ phức tạp Cách 3:** mọi thao tác `O(1)`; Space `O(n)` (mỗi nút thêm 2 con trỏ). Cách này dài và dễ sai hơn Cách 2, nhưng là bài luyện con trỏ rất tốt và dùng được ở ngôn ngữ không có `deque`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 bằng hàm `trace_ops` (in `L | R`, bất biến và kết quả):

| lệnh | L | R | len(L)−len(R) | trả |
|---|---|---|---|---|
| pushFront(1) | [1] | [] | 1 | null |
| pushBack(2) | [1] | [2] | 0 | null |
| pushMiddle(3) | [1,3] | [2] | 1 | null |
| pushMiddle(4) | [1,4] | [3,2] | 0 | null |
| popFront() | [4,3] | [2] | 1 | 1 |
| popMiddle() | [4] | [2] | 0 | 3 |
| popMiddle() | [2] | [] | 1 | 4 |
| popBack() | [] | [] | 0 | 2 |
| popFront() | [] | [] | 0 | −1 |

In [12]:
def trace_ops(ops, args):
    q = None
    out = []
    for op, arg in zip(ops, args):
        if op == "FrontMiddleBackQueue":
            q = QueueTwoDeque(); out.append(None); continue
        r = getattr(q, op)(*arg)
        out.append(r)
        diff = len(q.L) - len(q.R)
        assert diff in (0, 1)
        print(f"{op + '(' + ','.join(map(str, arg)) + ')':14} L={list(q.L)!s:10} R={list(q.R)!s:8} lệch={diff}  trả={r}")
    return out

assert trace_ops(EX_OPS, EX_ARGS) == EX_OUT

pushFront(1)   L=[1]        R=[]       lệch=1  trả=None
pushBack(2)    L=[1]        R=[2]      lệch=0  trả=None
pushMiddle(3)  L=[1, 3]     R=[2]      lệch=1  trả=None
pushMiddle(4)  L=[1, 4]     R=[3, 2]   lệch=0  trả=None
popFront()     L=[4, 3]     R=[2]      lệch=1  trả=1
popMiddle()    L=[4]        R=[2]      lệch=0  trả=3
popMiddle()    L=[2]        R=[]       lệch=1  trả=4
popBack()      L=[]         R=[]       lệch=0  trả=2
popFront()     L=[]         R=[]       lệch=0  trả=-1


## 9. Code nộp LeetCode

Đặt tên lớp đúng `FrontMiddleBackQueue` (cài đặt Cách 2):

In [13]:
class FrontMiddleBackQueue:

    def __init__(self):
        self.L = deque()
        self.R = deque()

    def _balance(self) -> None:
        if len(self.L) > len(self.R) + 1:
            self.R.appendleft(self.L.pop())
        elif len(self.R) > len(self.L):
            self.L.append(self.R.popleft())

    def pushFront(self, val: int) -> None:
        self.L.appendleft(val)
        self._balance()

    def pushMiddle(self, val: int) -> None:
        if len(self.L) > len(self.R):
            self.R.appendleft(self.L.pop())
        self.L.append(val)

    def pushBack(self, val: int) -> None:
        self.R.append(val)
        self._balance()

    def popFront(self) -> int:
        if not self.L:
            return -1
        val = self.L.popleft()
        self._balance()
        return val

    def popMiddle(self) -> int:
        if not self.L:
            return -1
        val = self.L.pop()
        self._balance()
        return val

    def popBack(self) -> int:
        if self.R:
            val = self.R.pop()
            self._balance()
            return val
        if self.L:
            return self.L.pop()
        return -1

got = run_ops(FrontMiddleBackQueue, EX_OPS, EX_ARGS)
print(got)
assert got == EX_OUT

[None, None, None, None, None, 1, 3, 4, 2, -1]


## 10. Vì sao thuật toán đúng

**Bất biến (I):** hàng đợi thật = `L + R` (nối theo thứ tự) và `0 ≤ len(L) − len(R) ≤ 1`.

**(I) ⇒ giữa phía trước là `L[-1]`:** đã chứng minh ở mục 6 cho cả `n` chẵn và lẻ. Đặc biệt `L` rỗng ⇔ hàng đợi rỗng (vì `len(L) ≥ len(R)`).

**Mỗi thao tác bảo toàn (I) và đúng ngữ nghĩa:**
- `pushFront`: `appendleft` vào `L` làm đúng "thêm đầu"; lệch tăng lên tối đa 2 ⇒ `_balance` chuyển `L[-1]` sang `R[0]` (không đổi thứ tự `L + R`) ⇒ lệch về 1.
- `pushBack`: tương tự, lệch có thể thành −1 ⇒ `_balance` chuyển `R[0]` về cuối `L`.
- `pushMiddle`: cần chèn vào chỉ số `n // 2`. Nếu lệch 0 (`n` chẵn) thì `n // 2 = len(L)` — ngay sau `L` ⇒ `L.append`, lệch thành 1. Nếu lệch 1 (`n` lẻ) thì `n // 2 = len(L) − 1` — ngay trước `L[-1]` ⇒ chuyển `L[-1]` sang đầu `R` (thứ tự không đổi) rồi `L.append`, lệch thành 0.
- `popFront`: phần tử đầu là `L[0]` (L khác rỗng khi hàng đợi khác rỗng) ⇒ đúng; lệch có thể thành −1 ⇒ cân bằng.
- `popMiddle`: lấy `L[-1]` đúng theo (I); lệch có thể thành −1 ⇒ cân bằng.
- `popBack`: phần tử cuối là `R[-1]` nếu `R` khác rỗng; nếu `R` rỗng thì theo (I) `len(L) ≤ 1` nên phần tử cuối là `L[-1]`.

`_balance` chỉ chuyển phần tử ở **chỗ nối** `L | R` nên không làm đổi thứ tự của `L + R`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số phần tử hiện có, `Q` là số lệnh (`Q ≤ 1000`).

| Cách | push/pop Front | push/pop Middle | push/pop Back | Tổng `Q` lệnh | Space |
|---|---|---|---|---|---|
| 1. list | `O(n)` | `O(n)` | `O(1)` | `O(Q²)` worst | `O(n)` |
| 2. Hai deque | `O(1)` | `O(1)` | `O(1)` | `O(Q)` | `O(n)` |
| 3. DLL + mid | `O(1)` | `O(1)` | `O(1)` | `O(Q)` | `O(n)` (hằng số lớn hơn) |

So sánh tốc độ trên chuỗi lệnh "giữa" dài:

In [14]:
for cls in (QueueList, QueueTwoDeque, QueueLinked):
    print(f"{cls.__name__:14}: {bench(cls, 10**5) * 1000:7.1f} ms cho 1e5 lệnh giữa")

QueueList     :   163.9 ms cho 1e5 lệnh giữa


QueueTwoDeque :    30.1 ms cho 1e5 lệnh giữa

QueueLinked   :    43.0 ms cho 1e5 lệnh giữa


## 12. Bộ test

Bộ test gồm: ví dụ đề, các chuỗi lệnh biên (pop khi rỗng, chỉ 1 phần tử, xen kẽ), và **chuỗi lệnh ngẫu nhiên** (seed cố định, `1 ≤ val ≤ 10⁹`, ≤ 1000 lệnh) đối chiếu từng kết quả với `QueueList` (brute force). Với hai cách nhanh còn kiểm tra cả **trạng thái** hàng đợi sau mỗi lệnh.

In [15]:
IMPLS = [QueueTwoDeque, QueueLinked, FrontMiddleBackQueue]

def state(q):
    if isinstance(q, QueueList): return q.a
    if isinstance(q, QueueLinked): return q.to_list()
    return list(q.L) + list(q.R)

def check_ops(ops, args, expected=None):
    ref = run_ops(QueueList, ops, args)
    if expected is not None:
        assert ref == expected
    for cls in IMPLS:
        assert run_ops(cls, ops, args) == ref, cls.__name__
    return ref

check_ops(EX_OPS, EX_ARGS, EX_OUT)

# edge case (tự dựng, đáp án tính tay và đối chiếu QueueList)
C = "FrontMiddleBackQueue"
check_ops([C, "popFront", "popMiddle", "popBack"], [[], [], [], []], [None, -1, -1, -1])
check_ops([C, "pushMiddle", "popBack"], [[], [7], []], [None, None, 7])
check_ops([C, "pushBack", "popFront"], [[], [7], []], [None, None, 7])
check_ops([C, "pushMiddle", "pushMiddle", "pushMiddle", "popMiddle", "popMiddle", "popMiddle"],
          [[], [1], [2], [3], [], [], []], [None, None, None, None, 3, 2, 1])
# ví dụ minh họa của đề: chèn 6 vào giữa [1,2,3,4,5]; lấy giữa [1,2,3,4,5,6]
q = FrontMiddleBackQueue()
for v in [1, 2, 3, 4, 5]: q.pushBack(v)
q.pushMiddle(6); assert state(q) == [1, 2, 6, 3, 4, 5]
q = FrontMiddleBackQueue()
for v in [1, 2, 3, 4, 5, 6]: q.pushBack(v)
assert q.popMiddle() == 3 and state(q) == [1, 2, 4, 5, 6]
print("ví dụ đề + edge case OK")

ví dụ đề + edge case OK


In [16]:
def random_ops(rnd, n_ops, p_push=0.55):
    names_push = ["pushFront", "pushMiddle", "pushBack"]
    names_pop = ["popFront", "popMiddle", "popBack"]
    ops, args = ["FrontMiddleBackQueue"], [[]]
    for _ in range(n_ops):
        if rnd.random() < p_push:
            ops.append(rnd.choice(names_push)); args.append([rnd.randint(1, 10**9)])
        else:
            ops.append(rnd.choice(names_pop)); args.append([])
    return ops, args

rnd = random.Random(2024)
for t in range(400):
    ops, args = random_ops(rnd, rnd.randint(1, 1000), p_push=rnd.choice([0.3, 0.5, 0.7]))
    check_ops(ops, args)

# kiểm tra trạng thái sau từng lệnh
rnd = random.Random(7)
for t in range(100):
    ops, args = random_ops(rnd, 300)
    objs = [QueueList()] + [cls() for cls in IMPLS]
    for op, arg in zip(ops[1:], args[1:]):
        res = [getattr(o, op)(*arg) for o in objs]
        assert len(set(res)) == 1
        st = [state(o) for o in objs]
        assert all(s == st[0] for s in st)
print("500 chuỗi lệnh ngẫu nhiên OK")

500 chuỗi lệnh ngẫu nhiên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm công thức chỉ số giữa khi **lấy ra** (dùng `n // 2`)

In [17]:
class WrongPopMiddle(QueueList):
    def popMiddle(self):
        return self.a.pop(len(self.a) // 2) if self.a else -1     # SAI với n chẵn

print("sai :", run_ops(WrongPopMiddle, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, None, None, None, None, 1, 3, 2, 4, -1]
đúng: [None, None, None, None, None, 1, 3, 4, 2, -1]


**Vì sao sai:** với `n` chẵn (ví dụ `[4,3,2,…]` có 4 phần tử) có hai vị trí giữa; đề chọn vị trí **phía trước** `(n−1)//2`, còn `n//2` là vị trí phía sau. **Sửa:** `pop((n - 1) // 2)` khi lấy ra, `insert(n // 2, v)` khi chèn.

### Lỗi 2 — Hai deque nhưng `pushMiddle` luôn `L.append` (quên trường hợp `n` lẻ)

In [18]:
class WrongPushMiddle(QueueTwoDeque):
    def pushMiddle(self, val):
        self.L.append(val)            # SAI khi len(L) > len(R)
        self._balance()

ops = ["FrontMiddleBackQueue", "pushBack", "pushBack", "pushBack", "pushMiddle", "popFront", "popFront", "popFront", "popFront"]
args = [[], [1], [2], [3], [9], [], [], [], []]
print("sai :", run_ops(WrongPushMiddle, ops, args))
print("đúng:", run_ops(QueueList, ops, args))

sai : [None, None, None, None, None, 1, 2, 9, 3]
đúng: [None, None, None, None, None, 1, 9, 2, 3]


**Vì sao sai:** với `[1,2,3]` thì `L = [1,2]`, `R = [3]`. Chèn giữa phải ra `[1,9,2,3]` (chỉ số `3 // 2 = 1`), nhưng `L.append(9)` rồi cân bằng lại cho `[1,2,9,3]`. **Sửa:** nếu `len(L) > len(R)` thì chuyển `L[-1]` sang đầu `R` **trước**, rồi mới `L.append(val)`.

### Lỗi 3 — `popBack` chỉ nhìn `R`, trả `-1` khi `R` rỗng

In [19]:
class WrongPopBack(QueueTwoDeque):
    def popBack(self):
        if not self.R:
            return -1                 # SAI: còn 1 phần tử nằm trong L
        v = self.R.pop(); self._balance(); return v

ops = ["FrontMiddleBackQueue", "pushFront", "popBack"]
args = [[], [5], []]
print("sai :", run_ops(WrongPopBack, ops, args))
print("đúng:", run_ops(QueueList, ops, args))

sai : [None, None, -1]
đúng: [None, None, 5]


**Vì sao sai:** theo bất biến, khi hàng đợi có đúng 1 phần tử thì nó nằm ở `L`, `R` rỗng. **Sửa:** nếu `R` rỗng mà `L` còn thì `L.pop()`.

### Lỗi 4 — Quên cân bằng sau `popFront`/`popMiddle`

In [20]:
class WrongNoBalance(QueueTwoDeque):
    def popFront(self):
        return self.L.popleft() if self.L else -1          # SAI: không _balance()

rnd = random.Random(1)
found = None
for _ in range(200):
    ops, args = random_ops(rnd, 30)
    if run_ops(WrongNoBalance, ops, args) != run_ops(QueueList, ops, args):
        found = (ops, args); break
ops, args = found
print("tìm được phản ví dụ dài", len(ops) - 1, "lệnh")
print("sai :", run_ops(WrongNoBalance, ops, args)[-10:])
print("đúng:", run_ops(QueueList, ops, args)[-10:])

tìm được phản ví dụ dài 30 lệnh
sai : [780055968, None, None, 67607931, None, 835284838, 645798692, 15620965, None, None]
đúng: [780055968, None, None, 645798692, None, 835284838, 67607931, -1, None, None]


**Vì sao sai:** sau `popFront`, `len(L)` có thể nhỏ hơn `len(R)` ⇒ bất biến hỏng ⇒ `L[-1]` không còn là phần tử giữa, và `L` rỗng không còn nghĩa là hàng đợi rỗng. Lỗi này **không** lộ ra ở ví dụ đề — chỉ test ngẫu nhiên mới bắt được. **Sửa:** gọi `_balance()` sau **mọi** thao tác có thể làm lệch.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thao tác ở **giữa** cấu trúc dữ liệu" ⇒ nghĩ ngay **chia đôi thành hai nửa** (hai deque, hai heap như LC 295, hai stack…). "Giữa" trở thành **chỗ nối**.
- Chốt một **bất biến kích thước** duy nhất và viết hàm `_balance()` dùng chung — đừng xử lý tay từng trường hợp.
- Chọn bất biến sao cho thao tác hay dùng nhất đơn giản nhất (ở đây `len(L) ≥ len(R)` ⇒ `popMiddle = L.pop()`).
- Bài design: **luôn** viết bộ chạy `run_ops` + cài đặt brute force bằng `list`, rồi bắn vài trăm chuỗi lệnh ngẫu nhiên.
- Công thức chỉ số giữa: lấy `(n−1)//2`, chèn `n//2` (cả hai đều "phía trước").

## 15. Biến thể của bài

1. **LC 641 — Design Circular Deque / LC 622 — Design Circular Queue:** hàng đợi hai đầu bằng mảng vòng.
2. **LC 295 — Find Median from Data Stream:** hai heap cân bằng, "giữa" là chỗ nối — cùng tư duy bất biến kích thước.
3. **LC 2296 — Design a Text Editor:** con trỏ ở giữa văn bản, dùng hai stack hai bên con trỏ.
4. **Giữa phía sau:** đổi quy ước chọn vị trí giữa **phía sau** khi có hai lựa chọn (bài tập 5).
5. **Thêm `peekFront/peekMiddle/peekBack`** không xóa (bài tập 4).
6. **Truy cập phần tử thứ `k` bất kỳ:** cần cấu trúc mạnh hơn (cây cân bằng / SortedList theo chỉ số, sqrt decomposition).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time mỗi lệnh | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. list | `insert`/`pop` theo công thức chỉ số | `O(n)` | `O(n)` | oracle, khi số lệnh nhỏ |
| 2. Hai deque | `L + R`, `0 ≤ len(L)−len(R) ≤ 1`, giữa = `L[-1]` | `O(1)` | `O(n)` | **nộp bài** |
| 3. DLL + mid | dịch con trỏ `mid` theo bảng chẵn/lẻ | `O(1)` | `O(n)` | luyện con trỏ, ngôn ngữ không có deque |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** chuỗi lệnh `pushBack(1), pushBack(2), pushBack(3), pushBack(4), popMiddle(), pushMiddle(9), popFront(), popMiddle()` trả về những gì (bỏ qua các `null`)?

**Bài 2 (tính tay):** Với Cách 2, sau các lệnh `pushFront(1), pushFront(2), pushFront(3), pushMiddle(4), pushBack(5)` thì `L` và `R` là gì?

**Bài 3 (điền TODO):** Hoàn thiện `balance(L, R)` khôi phục bất biến `0 ≤ len(L) − len(R) ≤ 1`, biết trước đó lệch tối đa 1 đơn vị so với khoảng cho phép.

**Bài 4 (tự viết lại):** Viết lớp `PeekQueue` kế thừa `FrontMiddleBackQueue`, thêm `peekFront()`, `peekMiddle()`, `peekBack()` (trả `-1` nếu rỗng, không xóa).

**Bài 5 (mini challenge):** Viết `BackMiddleQueue`: giống đề nhưng khi có hai vị trí giữa thì chọn vị trí **phía sau** (chèn vào chỉ số `(n+1)//2`, lấy ra chỉ số `n//2`). Mọi thao tác `O(1)`.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def balance(L, R):
    # TODO: nếu L dài hơn R quá 1 -> chuyển cuối L sang đầu R
    # TODO: nếu R dài hơn L -> chuyển đầu R sang cuối L
    pass

In [22]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class PeekQueue(FrontMiddleBackQueue):
    def peekFront(self):
        pass  # TODO
    def peekMiddle(self):
        pass  # TODO
    def peekBack(self):
        pass  # TODO

class BackMiddleQueue:
    def __init__(self):
        pass  # TODO

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: viết mảng sau mỗi lệnh. Mức 2: `[1,2,3,4]` lấy giữa chỉ số 1 → `2`; còn `[1,3,4]`, chèn 9 ở chỉ số 1 → `[1,9,3,4]`. Mức 3: kết quả `2, 1, 3`.

**Bài 2.** Mức 1: hàng đợi thật là `[3,2,1]` → `[3,4,2,1]` → `[3,4,2,1,5]`. Mức 2: `n = 5` ⇒ `len(L) = 3`. Mức 3: `L = [3,4,2]`, `R = [1,5]`.

**Bài 3.** Mức 1: hai nhánh `if/elif`. Mức 2: `R.appendleft(L.pop())`. Mức 3: `L.append(R.popleft())`.

**Bài 4.** Mức 1: giữa là `L[-1]`. Mức 2: đầu là `L[0]`; cuối là `R[-1]` nếu `R` khác rỗng, ngược lại `L[-1]`. Mức 3: trả `-1` khi `L` rỗng.

**Bài 5.** Mức 1: đổi vai: giữ `0 ≤ len(R) − len(L) ≤ 1` thì giữa phía sau luôn là `R[0]`. Mức 2: `popMiddle = R.popleft()`. Mức 3: `pushMiddle`: nếu `len(R) > len(L)` thì chuyển `R[0]` sang cuối `L` trước, rồi `R.appendleft(val)`. Đối chiếu bằng list với công thức `(n+1)//2` và `n//2`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
ops = ["FrontMiddleBackQueue", "pushBack", "pushBack", "pushBack", "pushBack", "popMiddle", "pushMiddle", "popFront", "popMiddle"]
args = [[], [1], [2], [3], [4], [], [9], [], []]
res = [x for x in run_ops(FrontMiddleBackQueue, ops, args) if x is not None]
print("Bài 1:", res)
assert res == [2, 1, 3] and res == [x for x in run_ops(QueueList, ops, args) if x is not None]

# Bài 2
q = FrontMiddleBackQueue()
q.pushFront(1); q.pushFront(2); q.pushFront(3); q.pushMiddle(4); q.pushBack(5)
print("Bài 2: L =", list(q.L), "R =", list(q.R))
assert list(q.L) == [3, 4, 2] and list(q.R) == [1, 5]

Bài 1: [2, 1, 3]
Bài 2: L = [3, 4, 2] R = [1, 5]


In [24]:
# Bài 3
def balance_sol(L, R):
    if len(L) > len(R) + 1:
        R.appendleft(L.pop())
    elif len(R) > len(L):
        L.append(R.popleft())

for a, b in [([1, 2, 3], [4]), ([1], [2, 3]), ([1, 2], [3]), ([], [1])]:
    L, R = deque(a), deque(b)
    balance_sol(L, R)
    assert list(L) + list(R) == a + b and 0 <= len(L) - len(R) <= 1
print("Bài 3 OK")

# Bài 4
class PeekQueueSol(FrontMiddleBackQueue):
    def peekFront(self):
        return self.L[0] if self.L else -1
    def peekMiddle(self):
        return self.L[-1] if self.L else -1
    def peekBack(self):
        if self.R: return self.R[-1]
        return self.L[-1] if self.L else -1

rnd = random.Random(11)
for _ in range(200):
    q, ref = PeekQueueSol(), QueueList()
    for _ in range(rnd.randint(1, 200)):
        op = rnd.choice(["pushFront", "pushMiddle", "pushBack", "popFront", "popMiddle", "popBack"])
        arg = [rnd.randint(1, 10**9)] if op.startswith("push") else []
        assert getattr(q, op)(*arg) == getattr(ref, op)(*arg)
        a = ref.a
        assert q.peekFront() == (a[0] if a else -1)
        assert q.peekMiddle() == (a[(len(a) - 1) // 2] if a else -1)
        assert q.peekBack() == (a[-1] if a else -1)
print("Bài 4 OK")

Bài 3 OK
Bài 4 OK


In [25]:
# Bài 5
class BackMiddleQueueSol:
    def __init__(self):
        self.L, self.R = deque(), deque()      # bất biến: 0 <= len(R) - len(L) <= 1
    def _balance(self):
        if len(self.R) > len(self.L) + 1:
            self.L.append(self.R.popleft())
        elif len(self.L) > len(self.R):
            self.R.appendleft(self.L.pop())
    def pushFront(self, val):
        self.L.appendleft(val); self._balance()
    def pushBack(self, val):
        self.R.append(val); self._balance()
    def pushMiddle(self, val):
        if len(self.R) > len(self.L):
            self.L.append(self.R.popleft())
        self.R.appendleft(val)
    def popFront(self):
        if self.L:
            v = self.L.popleft(); self._balance(); return v
        return self.R.popleft() if self.R else -1
    def popMiddle(self):
        if not self.R: return -1
        v = self.R.popleft(); self._balance(); return v
    def popBack(self):
        if not self.R: return -1
        v = self.R.pop(); self._balance(); return v

class BackMiddleList:
    def __init__(self): self.a = []
    def pushFront(self, v): self.a.insert(0, v)
    def pushBack(self, v): self.a.append(v)
    def pushMiddle(self, v): self.a.insert((len(self.a) + 1) // 2, v)
    def popFront(self): return self.a.pop(0) if self.a else -1
    def popMiddle(self): return self.a.pop(len(self.a) // 2) if self.a else -1
    def popBack(self): return self.a.pop() if self.a else -1

q = BackMiddleList()
for v in [1, 2, 3, 4, 5]: q.pushBack(v)
q.pushMiddle(6); print("chèn 6 (giữa phía sau) vào [1..5]:", q.a)
rnd = random.Random(12)
for _ in range(300):
    ops, args = random_ops(rnd, rnd.randint(1, 300))
    assert run_ops(BackMiddleQueueSol, ops, args) == run_ops(BackMiddleList, ops, args)
print("Bài 5 OK")

chèn 6 (giữa phía sau) vào [1..5]: [1, 2, 3, 6, 4, 5]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai nửa cân bằng" (giữa phía trước = `L[-1]`):**

```python
L, R = deque(), deque()            # bất biến: 0 <= len(L) - len(R) <= 1
def balance():
    if len(L) > len(R) + 1: R.appendleft(L.pop())
    elif len(R) > len(L):   L.append(R.popleft())

pushFront:  L.appendleft(v); balance()
pushBack:   R.append(v); balance()
pushMiddle: if len(L) > len(R): R.appendleft(L.pop())
            L.append(v)
popFront:   L.popleft() (nếu L rỗng -> -1); balance()
popMiddle:  L.pop()     (nếu L rỗng -> -1); balance()
popBack:    R.pop() nếu R; elif L: L.pop(); else -1; balance()
```

**Checklist:**
- [ ] Chỉ số giữa: lấy `(n−1)//2`, chèn `n//2` (giữa **phía trước**)?
- [ ] Mọi `pop*` trả `-1` khi hàng đợi rỗng?
- [ ] `popBack` xử lý trường hợp `R` rỗng nhưng `L` còn 1 phần tử?
- [ ] Gọi `_balance()` sau **mọi** thao tác có thể làm lệch?
- [ ] Tên lớp đúng `FrontMiddleBackQueue`, tên phương thức camelCase đúng như đề?
- [ ] Đã chạy vài trăm chuỗi lệnh ngẫu nhiên so với cài đặt `list`?